# Hybrid Retrieval Pipeline (BM25 + Dense)

This notebook implements an end-to-end **hybrid information retrieval** pipeline that combines:
- **BM25** (sparse / lexical) retrieval
- **Dense** (semantic / neural) retrieval via sentence-transformers

Multiple **fusion strategies** are benchmarked, including Reciprocal Rank Fusion (RRF) and divergence-weighted variants using Jensen–Shannon, KL, and Cross-Entropy divergences.

**How to use:**
1. Edit `config.yaml` to add or remove dataset names under the `datasets:` key.
2. Run all cells — the pipeline automatically detects which steps are already complete and skips them.
3. Results and charts are displayed at the end.

## 1 · Imports & Configuration

Load all required libraries and the project configuration from `config.yaml`.  
Utility functions (I/O, serialization, state management) live in `utils.py`.

In [ ]:
import os
import gc
import json
import csv
import warnings
import numpy as np
import torch
from tqdm.auto import tqdm
from collections import Counter
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import ndcg_score
from nltk.stem import SnowballStemmer
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util as st_util
from beir import util as beir_util
from beir.datasets.data_loader import GenericDataLoader

warnings.filterwarnings("ignore", category=FutureWarning)

# Project utilities (I/O helpers, state management, etc.)
from utils import (
    load_config, ensure_dir, count_lines,
    save_pickle, load_pickle, load_queries, load_qrels,
    initialize_output_files,
    append_corpus_to_jsonl, append_queries_to_jsonl, append_qrels_to_tsv,
    load_corpus_batch_generator,
    get_datasets_hash, load_pipeline_state, save_pipeline_state,
    is_step_complete, mark_step_complete, clear_pipeline_artifacts,
)

# ── Load config ──
config = load_config()
paths = config["paths"]
datasets_to_merge = config.get("datasets", [])

print(f"Active datasets : {datasets_to_merge}")
print(f"Embedding model : {config['embeddings']['model_name']}")
print(f"Device          : {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2 · Pipeline State & Change Detection

A lightweight JSON state file tracks which processing steps have already been completed.  
If the dataset list in `config.yaml` changes (datasets added or removed), **all previous artifacts are cleared** and the pipeline rebuilds from scratch.

In [ ]:
# ── State management ──
state_path = paths["pipeline_state"]
state = load_pipeline_state(state_path)

current_hash = get_datasets_hash(datasets_to_merge)
previous_hash = state.get("datasets_hash")

if previous_hash is not None and previous_hash != current_hash:
    print("⚠  Dataset list changed — clearing all cached artifacts for a clean rebuild...")
    n = clear_pipeline_artifacts(paths, state_path)
    print(f"   Removed {n} file(s).")
    state = load_pipeline_state(state_path)  # reload after reset

state["datasets_hash"] = current_hash
save_pipeline_state(state, state_path)

print(f"Datasets hash   : {current_hash}")
print(f"Completed steps : {state.get('completed_steps', [])}")

## 3 · Download & Merge BEIR Datasets

For every dataset listed in `config.yaml`:
1. Download it from the BEIR public repository (if not already on disk).
2. Load its corpus, queries, and relevance judgments.
3. Append everything into unified JSONL/TSV files (IDs are prefixed with the dataset name to avoid collisions).

**Skipped** if the merged files already exist and the dataset list hasn't changed.

In [ ]:
STEP = "download_and_merge"
required_files = [paths["corpus"], paths["queries"], paths["qrels"]]

if is_step_complete(state, STEP, required_files):
    print(f"✓ Step '{STEP}' already complete — skipping.")
else:
    data_folder = paths["datasets_folder"]
    ensure_dir(data_folder)
    ensure_dir(paths["data_folder"])

    initialize_output_files(paths["corpus"], paths["queries"], paths["qrels"])

    for dataset_name in datasets_to_merge:
        print(f"\nProcessing: {dataset_name}")
        dataset_path = os.path.join(data_folder, dataset_name)

        # Download if missing
        if not os.path.exists(dataset_path):
            print(f"  Downloading {dataset_name}...")
            url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset_name}.zip"
            try:
                beir_util.download_and_unzip(url, data_folder)
            except Exception as e:
                print(f"  ✗ Download failed: {e}")
                continue

        # Load the best available split
        try:
            loader = GenericDataLoader(dataset_path)
            split = None
            for candidate in ["test", "train", "dev"]:
                if os.path.exists(os.path.join(dataset_path, "qrels", f"{candidate}.tsv")):
                    split = candidate
                    break
            if split is None:
                print(f"  ✗ No valid split found — skipping.")
                continue
            corpus, queries, qrels_data = loader.load(split=split)
            print(f"  Loaded split '{split}': {len(corpus)} docs, {len(queries)} queries")
        except Exception as e:
            print(f"  ✗ Load failed: {e}")
            continue

        append_corpus_to_jsonl(corpus, paths["corpus"], dataset_name)
        append_queries_to_jsonl(queries, paths["queries"], dataset_name)
        append_qrels_to_tsv(qrels_data, paths["qrels"], dataset_name)

        del corpus, queries, qrels_data
        gc.collect()

    mark_step_complete(state, STEP, state_path)
    print("\n✓ Merge complete.")

### Merged corpus statistics

In [ ]:
n_docs    = count_lines(paths["corpus"])
n_queries = count_lines(paths["queries"])
n_qrels   = count_lines(paths["qrels"]) - 1  # header row

print(f"Documents : {n_docs:,}")
print(f"Queries   : {n_queries:,}")
print(f"Qrels     : {n_qrels:,}")

## 4 · Preprocess Corpus (Tokenization & Stemming)

Each document's title + text is lowercased, split on whitespace, and stemmed with NLTK's Snowball stemmer.  
The result is saved as a JSONL of `{_id, tokens}` for BM25 indexing.

**Skipped** if the tokenized corpus file already exists.

In [ ]:
STEP = "preprocess_corpus"
required_files = [paths["tokenized_corpus"]]

if is_step_complete(state, STEP, required_files):
    print(f"✓ Step '{STEP}' already complete — skipping.")
else:
    stemmer = SnowballStemmer(config["preprocessing"]["stemmer_language"])
    corpus_path = paths["corpus"]
    output_path = paths["tokenized_corpus"]
    total = count_lines(corpus_path)

    with open(output_path, "w", encoding="utf-8") as f_out, \
         open(corpus_path, "r", encoding="utf-8") as f_in:
        for line in tqdm(f_in, total=total, desc="Stemming documents"):
            doc = json.loads(line)
            raw = (doc.get("title", "") + " " + doc.get("text", "")).strip()
            tokens = [stemmer.stem(t) for t in raw.lower().split()]
            json.dump({"_id": doc["_id"], "tokens": tokens}, f_out)
            f_out.write("\n")

    mark_step_complete(state, STEP, state_path)
    print(f"✓ Tokenized corpus saved ({total:,} documents).")

## 5 · Build Frequency Index

Count stemmed token occurrences across the whole corpus.  
This is used later to compute query–corpus divergence for adaptive fusion weights.

In [ ]:
STEP = "build_freq_index"
required_files = [paths["freq_index"]]

if is_step_complete(state, STEP, required_files):
    print(f"✓ Step '{STEP}' already complete — skipping.")
else:
    global_counter = Counter()
    total_tokens = 0

    with open(paths["tokenized_corpus"], "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Counting tokens"):
            entry = json.loads(line)
            global_counter.update(entry["tokens"])
            total_tokens += len(entry["tokens"])

    save_pickle({"counts": dict(global_counter), "total_tokens": total_tokens}, paths["freq_index"])
    mark_step_complete(state, STEP, state_path)
    print(f"✓ Frequency index saved — {len(global_counter):,} unique stems, {total_tokens:,} total tokens.")

## 6 · Build BM25 Index

Create an Okapi-BM25 index from the tokenized corpus using `rank_bm25`.

In [ ]:
STEP = "build_bm25_index"
required_files = [paths["bm25_index"]]

if is_step_complete(state, STEP, required_files):
    print(f"✓ Step '{STEP}' already complete — skipping.")
else:
    tokenized_corpus = []
    doc_ids = []

    with open(paths["tokenized_corpus"], "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Loading tokenized docs"):
            entry = json.loads(line)
            tokenized_corpus.append(entry["tokens"])
            doc_ids.append(entry["_id"])

    print(f"Building BM25 model over {len(doc_ids):,} documents...")
    bm25 = BM25Okapi(tokenized_corpus)
    save_pickle({"model": bm25, "doc_ids": doc_ids}, paths["bm25_index"])

    mark_step_complete(state, STEP, state_path)
    print(f"✓ BM25 index saved.")

## 7 · Build Dense Embeddings (Corpus)

Encode every document with the configured sentence-transformer model.  
Embeddings are generated in batches and saved as a single `torch.Tensor`.

In [ ]:
STEP = "build_embeddings"
required_files = [paths["corpus_embeddings"], paths["vector_doc_ids"]]

if is_step_complete(state, STEP, required_files):
    print(f"✓ Step '{STEP}' already complete — skipping.")
else:
    emb_cfg = config["embeddings"]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer(emb_cfg["model_name"], device=device)

    total_docs = count_lines(paths["corpus"])
    batch_size = emb_cfg["batch_size"]

    all_embeddings = []
    all_doc_ids = []

    loader = load_corpus_batch_generator(paths["corpus"], batch_size)
    for batch_ids, batch_texts in tqdm(loader, total=total_docs // batch_size + 1, desc="Encoding corpus"):
        embs = model.encode(batch_texts, convert_to_tensor=True, show_progress_bar=False)
        all_embeddings.append(embs.cpu())
        all_doc_ids.extend(batch_ids)

    final_tensor = torch.cat(all_embeddings, dim=0)
    print(f"Embedding matrix shape: {final_tensor.shape}")

    ensure_dir(os.path.dirname(paths["corpus_embeddings"]))
    torch.save(final_tensor, paths["corpus_embeddings"])
    save_pickle(all_doc_ids, paths["vector_doc_ids"])

    del model, all_embeddings, final_tensor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    mark_step_complete(state, STEP, state_path)
    print(f"✓ Corpus embeddings saved ({len(all_doc_ids):,} vectors).")

## 8 · Tokenize Queries

Stem and tokenize queries the same way we processed the corpus, so BM25 scoring is consistent.

In [ ]:
STEP = "tokenize_queries"
required_files = [paths["tokenized_queries"]]

if is_step_complete(state, STEP, required_files):
    print(f"✓ Step '{STEP}' already complete — skipping.")
else:
    queries = load_queries(paths["queries"])
    stemmer = SnowballStemmer(config["preprocessing"]["stemmer_language"])

    tokenized_queries = {}
    for qid, text in tqdm(queries.items(), desc="Stemming queries"):
        tokenized_queries[qid] = [stemmer.stem(t) for t in text.lower().split()]

    save_pickle(tokenized_queries, paths["tokenized_queries"])
    mark_step_complete(state, STEP, state_path)
    print(f"✓ Tokenized {len(tokenized_queries):,} queries.")

## 9 · BM25 Retrieval

Score every query against the BM25 index and collect top-k document rankings.

In [ ]:
STEP = "build_bm25_results"
required_files = [paths["bm25_results"]]
top_k = config["benchmark"]["top_k"]

if is_step_complete(state, STEP, required_files):
    print(f"✓ Step '{STEP}' already complete — skipping.")
else:
    bm25_data = load_pickle(paths["bm25_index"])
    bm25_model = bm25_data["model"]
    bm25_doc_map = bm25_data["doc_ids"]
    tokenized_queries = load_pickle(paths["tokenized_queries"])

    bm25_results = {}
    bm25_rankings = {}

    for qid, tokens in tqdm(tokenized_queries.items(), desc="BM25 search"):
        scores = bm25_model.get_scores(tokens)
        top_idx = np.argsort(scores)[::-1][:top_k]
        bm25_results[qid] = {}
        ranking = []
        for idx in top_idx:
            doc_id = bm25_doc_map[idx]
            bm25_results[qid][doc_id] = float(scores[idx])
            ranking.append(doc_id)
        bm25_rankings[qid] = ranking

    save_pickle({"results": bm25_results, "rankings": bm25_rankings}, paths["bm25_results"])

    del bm25_data, bm25_model
    gc.collect()

    mark_step_complete(state, STEP, state_path)
    print(f"✓ BM25 results saved for {len(bm25_results):,} queries (top-{top_k}).")

## 10 · Dense Query Vectors

Encode queries with the same sentence-transformer used for the corpus.

In [ ]:
STEP = "build_dense_query_vectors"
required_files = [paths["dense_query_vectors"], paths["dense_query_ids"]]

if is_step_complete(state, STEP, required_files):
    print(f"✓ Step '{STEP}' already complete — skipping.")
else:
    queries = load_queries(paths["queries"])
    query_ids = list(queries.keys())
    query_texts = [queries[qid] for qid in query_ids]

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer(config["embeddings"]["model_name"], device=device)

    print(f"Encoding {len(query_ids):,} queries...")
    query_emb = model.encode(query_texts, convert_to_tensor=True, show_progress_bar=True)

    ensure_dir(os.path.dirname(paths["dense_query_vectors"]))
    torch.save(query_emb.cpu(), paths["dense_query_vectors"])
    save_pickle(query_ids, paths["dense_query_ids"])

    del model, query_emb
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    mark_step_complete(state, STEP, state_path)
    print(f"✓ Dense query vectors saved ({len(query_ids):,} queries).")

## 11 · Dense Retrieval (Semantic Search)

Perform cosine-similarity search between query embeddings and corpus embeddings using `sentence_transformers.util.semantic_search`.

In [ ]:
STEP = "build_dense_results"
required_files = [paths["dense_results"]]

if is_step_complete(state, STEP, required_files):
    print(f"✓ Step '{STEP}' already complete — skipping.")
else:
    search_cfg = config["dense_search"]

    corpus_emb = torch.load(paths["corpus_embeddings"], map_location="cpu")
    vector_doc_map = load_pickle(paths["vector_doc_ids"])
    query_emb = torch.load(paths["dense_query_vectors"], map_location="cpu")
    query_ids = load_pickle(paths["dense_query_ids"])

    print(f"Semantic search: {len(query_ids):,} queries × {corpus_emb.shape[0]:,} docs (top-{top_k})...")
    hits_list = st_util.semantic_search(
        query_emb, corpus_emb,
        top_k=top_k,
        query_chunk_size=search_cfg["query_chunk_size"],
        corpus_chunk_size=search_cfg["corpus_chunk_size"],
    )

    dense_results = {}
    dense_rankings = {}
    for i, hits in enumerate(hits_list):
        qid = query_ids[i]
        dense_results[qid] = {}
        dense_rankings[qid] = []
        for hit in hits:
            doc_id = vector_doc_map[hit["corpus_id"]]
            dense_results[qid][doc_id] = float(hit["score"])
            dense_rankings[qid].append(doc_id)

    save_pickle({"results": dense_results, "rankings": dense_rankings}, paths["dense_results"])

    del corpus_emb, query_emb
    gc.collect()

    mark_step_complete(state, STEP, state_path)
    print(f"✓ Dense results saved for {len(dense_results):,} queries (top-{top_k}).")

## 12 · Fusion Methods

Core algorithmic functions for combining sparse and dense retrieval results:

| Function | Description |
|----------|-------------|
| `sigmoid` | Shifted & scaled sigmoid for smooth alpha mapping |
| `get_query_distribution` | Computes P(w|Q) and P(w|C) distributions over unique query terms |
| `compute_divergence_alpha` | Derives per-query dense weight α from JSD / KLD / CE divergence |
| `apply_rrf_fusion` | Standard Reciprocal Rank Fusion |
| `apply_wrrf_fusion` | Weighted RRF — scales each list's contribution by α |
| `get_sorted_docs` | Sort documents by descending score |
| `calculate_ndcg_at_k` | NDCG@k evaluation for a single query |

In [ ]:
# ═══════════════════════════════════════════
# Fusion & Evaluation Functions
# ═══════════════════════════════════════════

def sigmoid(x, k=10, x0=0):
    """Standard sigmoid, shifted by x0 and scaled by k."""
    return 1.0 / (1.0 + np.exp(-k * (x - x0)))


def get_query_distribution(query_tokens, global_counts, total_tokens):
    """Return P(w|Q) and P(w|C) arrays over unique query terms.

    Uses add-one smoothing for terms absent from the corpus so that
    KL / JS divergence remains well-defined.
    """
    q_len = len(query_tokens)
    if q_len == 0:
        return None, None

    term_freq = Counter(query_tokens)
    unique_tokens = list(term_freq.keys())

    p_corpus = np.empty(len(unique_tokens))
    p_query  = np.empty(len(unique_tokens))

    for i, t in enumerate(unique_tokens):
        count = global_counts.get(t, 0)
        if count == 0:
            count = 1  # add-one smoothing
        p_corpus[i] = count / total_tokens
        p_query[i]  = term_freq[t] / q_len

    return p_query, p_corpus


def compute_divergence_alpha(query_tokens, freq_data, config,
                             method="jsd", use_sigmoid=False,
                             normalize_01=False, div_stats=None):
    """Compute the dense-weight alpha from query–corpus divergence.

    High divergence → query is specific → lean towards BM25 (low α).
    Low divergence  → query is generic  → lean towards dense (high α).
    """
    p_q, p_c = get_query_distribution(
        query_tokens, freq_data["counts"], freq_data["total_tokens"]
    )
    if p_q is None:
        return 0.5

    if "jsd" in method:
        div_score = jensenshannon(p_q, p_c, base=2) ** 2
    elif "kld" in method:
        div_score = np.sum(p_q * np.log(p_q / p_c))
    elif "crossentropy" in method or "ce" in method:
        div_score = -np.sum(p_q * np.log(p_c))
    else:
        div_score = 0.5

    # Optional [0, 1] normalization
    if normalize_01 and div_stats is not None:
        d_min, d_max = div_stats["min"], div_stats["max"]
        if d_max > d_min:
            div_score = (div_score - d_min) / (d_max - d_min)
        else:
            div_score = 0.5

    if use_sigmoid:
        params = config["benchmark"]["sigmoid"]
        alpha = sigmoid(params["center"] - div_score, k=params["slope"])
    else:
        alpha = 1.0 - div_score

    return float(np.clip(alpha, 0.0, 1.0))


def apply_rrf_fusion(sparse_rankings, dense_rankings, k_rrf):
    """Reciprocal Rank Fusion over two ranked lists."""
    fused = {}
    for rank, doc in enumerate(sparse_rankings):
        fused[doc] = fused.get(doc, 0.0) + 1.0 / (k_rrf + rank + 1)
    for rank, doc in enumerate(dense_rankings):
        fused[doc] = fused.get(doc, 0.0) + 1.0 / (k_rrf + rank + 1)
    return fused


def apply_wrrf_fusion(sparse_rankings, dense_rankings, alpha, k_rrf):
    """Weighted RRF — alpha scales the dense list, (1-alpha) the sparse list."""
    fused = {}
    for rank, doc in enumerate(sparse_rankings):
        fused[doc] = fused.get(doc, 0.0) + (1.0 - alpha) / (k_rrf + rank + 1)
    for rank, doc in enumerate(dense_rankings):
        fused[doc] = fused.get(doc, 0.0) + alpha / (k_rrf + rank + 1)
    return fused


def get_sorted_docs(scores_dict):
    """Return doc IDs sorted by descending score."""
    return sorted(scores_dict, key=scores_dict.get, reverse=True)


def calculate_ndcg_at_k(result_ids, qrels, k=10):
    """NDCG@k for a single query given ranked doc IDs and ground-truth relevance."""
    top_hits = result_ids[:k]
    if not top_hits:
        return 0.0
    predicted = [qrels.get(doc, 0) for doc in top_hits]
    ideal = sorted(qrels.values(), reverse=True)[:k]
    if len(ideal) < k:
        ideal += [0] * (k - len(ideal))
    return ndcg_score([ideal], [predicted], k=k)

print("✓ Fusion & evaluation functions defined.")

## 13 · Retrieval Benchmark

Evaluate **9 retrieval strategies** and collect NDCG@k metrics:

| # | Strategy | Description |
|---|----------|-------------|
| 1 | BM25 Only | Sparse baseline |
| 2 | Dense Only | Semantic baseline |
| 3 | Naive RRF | Equal-weight rank fusion |
| 4 | JSD (Linear) | Jensen–Shannon divergence → linear α |
| 5 | JSD (Sigmoid) | JSD → sigmoid-mapped α |
| 6 | KLD (0-1 Norm) | Kullback–Leibler divergence, min-max normalized |
| 7 | KLD (0-1 + Sigmoid) | KLD normalized + sigmoid |
| 8 | CE (0-1 Norm) | Cross-Entropy, min-max normalized |
| 9 | CE (0-1 + Sigmoid) | CE normalized + sigmoid |

In [ ]:
# ── Load all pre-computed data ──
qrels = load_qrels(paths["qrels"])
bench = config["benchmark"]
ndcg_k = bench["ndcg_k"]
k_rrf  = bench["rrf"]["k"]

bm25_data = load_pickle(paths["bm25_results"])
sparse_rankings = bm25_data["rankings"]

dense_data = load_pickle(paths["dense_results"])
dense_rankings = dense_data["rankings"]

tokenized_queries = load_pickle(paths["tokenized_queries"])
freq_data = load_pickle(paths["freq_index"])

query_ids = [qid for qid in sparse_rankings if qid in qrels]
print(f"Evaluating {len(query_ids):,} queries  |  NDCG@{ndcg_k}  |  top-{top_k}")

### Pre-compute divergence statistics

We need the min/max divergence values across all queries for the normalized methods.

In [ ]:
div_stats = {
    "jsd": {"values": []},
    "kld": {"values": []},
    "ce":  {"values": []},
}

for qid in query_ids:
    p_q, p_c = get_query_distribution(
        tokenized_queries[qid], freq_data["counts"], freq_data["total_tokens"]
    )
    if p_q is not None:
        div_stats["jsd"]["values"].append(jensenshannon(p_q, p_c, base=2) ** 2)
        div_stats["kld"]["values"].append(float(np.sum(p_q * np.log(p_q / p_c))))
        div_stats["ce"]["values"].append(float(-np.sum(p_q * np.log(p_c))))

for key in div_stats:
    vals = div_stats[key]["values"]
    div_stats[key]["min"] = float(np.min(vals))
    div_stats[key]["max"] = float(np.max(vals))
    print(f"  {key.upper()} range: [{div_stats[key]['min']:.4f}, {div_stats[key]['max']:.4f}]")

### Run all fusion strategies

In [ ]:
methods = [
    ("BM25 Only",                "bm25_only"),
    ("Dense Only",               "dense_only"),
    ("Naive RRF",                "rrf"),
    ("JSD (Linear)",             "jsd_linear"),
    ("JSD (Sigmoid)",            "jsd_sigmoid"),
    ("KLD (0-1 Norm)",           "kld_normalized"),
    ("KLD (0-1 + Sigmoid)",      "kld_normalized_sigmoid"),
    ("Cross-Entropy (0-1 Norm)", "ce_normalized"),
    ("Cross-Entropy (0-1 + Sigmoid)", "ce_normalized_sigmoid"),
]

final_metrics = []

for name, mode in methods:
    ndcg_scores = []
    alpha_values = []

    for qid in tqdm(query_ids, desc=f"Eval {name}", leave=False):
        s_rank = sparse_rankings[qid]
        d_rank = dense_rankings[qid]

        if mode == "bm25_only":
            sorted_docs = s_rank
            alpha_values.append(0.0)

        elif mode == "dense_only":
            sorted_docs = d_rank
            alpha_values.append(1.0)

        elif mode == "rrf":
            fused = apply_rrf_fusion(s_rank, d_rank, k_rrf)
            sorted_docs = get_sorted_docs(fused)
            alpha_values.append(0.5)

        else:
            use_sig  = "sigmoid" in mode
            use_norm = "normalized" in mode
            metric   = "ce" if "ce" in mode else ("kld" if "kld" in mode else "jsd")

            alpha = compute_divergence_alpha(
                tokenized_queries[qid], freq_data, config,
                method=metric, use_sigmoid=use_sig,
                normalize_01=use_norm,
                div_stats=div_stats[metric] if use_norm else None,
            )
            alpha_values.append(alpha)
            fused = apply_wrrf_fusion(s_rank, d_rank, alpha, k_rrf)
            sorted_docs = get_sorted_docs(fused)

        score = calculate_ndcg_at_k(sorted_docs, qrels[qid], k=ndcg_k)
        ndcg_scores.append(score)

    avg_ndcg  = float(np.mean(ndcg_scores))
    avg_alpha = float(np.mean(alpha_values))

    print(f"  {name:<35s}  NDCG@{ndcg_k} = {avg_ndcg:.4f}   α_avg = {avg_alpha:.3f}")

    final_metrics.append({
        "Method": name,
        f"NDCG@{ndcg_k}": avg_ndcg,
        "Min_Alpha": float(np.min(alpha_values)),
        "Max_Alpha": float(np.max(alpha_values)),
        "Avg_Alpha": avg_alpha,
    })

## 14 · Save Results

In [ ]:
results_path = paths["results"]
ensure_dir(os.path.dirname(results_path))

fieldnames = ["Method", f"NDCG@{ndcg_k}", "Min_Alpha", "Max_Alpha", "Avg_Alpha"]
with open(results_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(final_metrics)

print(f"✓ Results saved to {results_path}")

## 15 · Visualization

Bar chart comparing NDCG@k across all fusion strategies, plus an alpha-range plot.

In [ ]:
try:
    import matplotlib
    import matplotlib.pyplot as plt
    matplotlib.rcParams.update({"figure.dpi": 120, "font.size": 10})
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("matplotlib not installed — skipping plots. Install with: pip install matplotlib")

if HAS_MPL:
    names  = [m["Method"] for m in final_metrics]
    scores = [m[f"NDCG@{ndcg_k}"] for m in final_metrics]
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(names)))

    # ── NDCG bar chart ──
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.barh(names[::-1], scores[::-1], color=colors[::-1], edgecolor="white", linewidth=0.5)
    ax.set_xlabel(f"NDCG@{ndcg_k}")
    ax.set_title(f"Retrieval Performance — NDCG@{ndcg_k}")
    ax.set_xlim(0, max(scores) * 1.12)

    for bar, val in zip(bars, scores[::-1]):
        ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=9)

    plt.tight_layout()
    plt.show()

    # ── Alpha range chart ──
    fusion_metrics = [m for m in final_metrics if m["Avg_Alpha"] not in (0.0, 0.5, 1.0)]
    if fusion_metrics:
        fig2, ax2 = plt.subplots(figsize=(10, 4))
        f_names = [m["Method"] for m in fusion_metrics]
        mins    = [m["Min_Alpha"] for m in fusion_metrics]
        maxs    = [m["Max_Alpha"] for m in fusion_metrics]
        avgs    = [m["Avg_Alpha"] for m in fusion_metrics]

        y_pos = np.arange(len(f_names))
        for i in range(len(f_names)):
            ax2.plot([mins[i], maxs[i]], [y_pos[i], y_pos[i]], "o-", color="steelblue", markersize=5)
            ax2.plot(avgs[i], y_pos[i], "D", color="tomato", markersize=7, zorder=5)

        ax2.set_yticks(y_pos)
        ax2.set_yticklabels(f_names)
        ax2.set_xlabel("Alpha (dense weight)")
        ax2.set_title("Per-query Alpha Range  (● min/max,  ◆ mean)")
        ax2.set_xlim(-0.05, 1.05)
        ax2.grid(axis="x", alpha=0.3)
        plt.tight_layout()
        plt.show()

## 16 · Summary Table

In [ ]:
# Pretty-print results table
header = f"{'Method':<35s}  {'NDCG@'+str(ndcg_k):>10s}  {'α_min':>7s}  {'α_max':>7s}  {'α_avg':>7s}"
print(header)
print("─" * len(header))
for m in final_metrics:
    print(f"{m['Method']:<35s}  {m[f'NDCG@{ndcg_k}']:>10.4f}  {m['Min_Alpha']:>7.3f}  {m['Max_Alpha']:>7.3f}  {m['Avg_Alpha']:>7.3f}")

best = max(final_metrics, key=lambda x: x[f"NDCG@{ndcg_k}"])
print(f"\n★  Best method: {best['Method']}  →  NDCG@{ndcg_k} = {best[f'NDCG@{ndcg_k}']:.4f}")